In [1]:
from langgraph.graph import START, StateGraph,MessagesState
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
llm1 = ChatOllama(model="nemotron-3-nano:30b-cloud")
llm2 = ChatGroq(model="llama-3.3-70b-versatile")
llm3 = ChatOllama(model="smollm2:360m")

In [3]:
final_model = llm1.with_fallbacks(fallbacks=[llm2,llm3])

In [4]:
def call_model(state: MessagesState):
    response= final_model.invoke(state["messages"])
    return{
        "messages":[response]
    }

In [5]:
workflow = StateGraph(MessagesState)
workflow.add_node(call_model,"call_model")
workflow.add_edge(START,"call_model")

In [6]:
DB_URL= os.getenv("DB_URL")
if not DB_URL:
    raise ValueError("DATABASE_URL not found in environment or .env file!")

In [7]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # Run once (creates tables)
    checkpointer.setup()

    graph= workflow.compile(checkpointer=checkpointer)

    # Thread -1 
    t1 = {
        "configurable": {
        "thread_id":"thread-1"
        }
    }
    graph.invoke({
        "messages":[
            {
                "role":"user",
                "content":"Hello my name is Suman"
            }
        ]
    },t1)
    out1 = graph.invoke({
        "messages":[
            {
                "role":"user",
                "content":"What is my name"
            }
        ]
    },t1)

    print("Thread-1",out1["messages"][-1].content)

Thread-1 Your name is **Suman**! 🌟 If there’s anything specific you’d like to discuss or explore, just let me know.
